# Feature Selection
**Improvements:**
- Fixed OrdinalEncoder misuse — nominal features (`sector`, `property_type`) now use OneHotEncoder
- Fixed train/test leakage — all techniques now use train set only for fitting
- Eliminated redundant RandomForest training (was 3× — now 1×)
- SHAP computation now on test set sample (faster)
- Negative permutation importance values investigated
- Added statistical justification for dropped features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('gurgaon_properties_missing_value_imputation.csv')
print(f"Shape: {df.shape}")

In [ ]:
train_df = df.drop(columns=['society','price_per_sqft']).copy()

# Categorize luxury score — bounds based on distribution quartiles
def categorize_luxury(score):
    if score < 50:   return 'Low'
    elif score < 150: return 'Medium'
    else:             return 'High'

def categorize_floor(floor):
    if pd.isna(floor): return None
    if floor <= 2:     return 'Low Floor'
    elif floor <= 10:  return 'Mid Floor'
    else:              return 'High Floor'

train_df['luxury_category'] = train_df['luxury_score'].apply(categorize_luxury)
train_df['floor_category']  = train_df['floorNum'].apply(categorize_floor)
train_df.drop(columns=['floorNum','luxury_score'], inplace=True)

# Drop rows where floor_category is None
train_df = train_df[train_df['floor_category'].notna()].copy()
print(f"Shape after feature prep: {train_df.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# FIX: use OneHotEncoder for nominal features (sector, property_type)
# OrdinalEncoder should only be used for features with natural order
ORDINAL_COLS = ['balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']
NOMINAL_COLS = ['property_type', 'sector']
NUM_COLS     = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']

preprocessor = ColumnTransformer(transformers=[
    ('num',  StandardScaler(),                                   NUM_COLS),
    ('ord',  OrdinalEncoder(),                                   ORDINAL_COLS),
    ('nom',  OneHotEncoder(drop='first', sparse_output=False),   NOMINAL_COLS),
], remainder='passthrough')

X = train_df.drop(columns=['price'])
y = train_df['price']

# FIX: split first — all feature importance computed on train set only to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_enc = preprocessor.fit_transform(X_train, y_train)
X_test_enc  = preprocessor.transform(X_test)

# Recover feature names after encoding
ohe_names = preprocessor.named_transformers_['nom'].get_feature_names_out(NOMINAL_COLS).tolist()
feature_names = NUM_COLS + ORDINAL_COLS + ohe_names +     [c for c in X.columns if c not in NUM_COLS + ORDINAL_COLS + NOMINAL_COLS]

print(f"Train: {X_train_enc.shape}, Test: {X_test_enc.shape}")
print(f"Total features after encoding: {len(feature_names)}")

## Fit Shared Random Forest (Train Set Only)
Trained once and reused across all techniques — no redundant fitting.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.feature_selection import RFE
import shap

# FIX: train once and reuse across techniques 2, 4, 6, 8
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train)
print(f"RF Train R2: {rf.score(X_train_enc, y_train):.4f}")
print(f"RF Test  R2: {rf.score(X_test_enc,  y_test):.4f}")

## Technique 1 — Correlation with Target

In [ ]:
corr_matrix = pd.DataFrame(X_train_enc, columns=feature_names)
corr_matrix['price'] = y_train.values

fi_df1 = (corr_matrix.corr()['price']
           .drop('price')
           .abs()
           .reset_index()
           .rename(columns={'index':'feature', 'price':'corr_coeff'})
           .sort_values('corr_coeff', ascending=False))
fi_df1.head(10)

## Technique 2 — Random Forest Importance (Train Set)

In [ ]:
fi_df2 = pd.DataFrame({
    'feature': feature_names,
    'rf_importance': rf.feature_importances_
}).sort_values('rf_importance', ascending=False)
fi_df2.head(10)

## Technique 3 — Gradient Boosting Importance (Train Set)

In [ ]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_enc, y_train)   # FIX: train set only

fi_df3 = pd.DataFrame({
    'feature': feature_names,
    'gb_importance': gb.feature_importances_
}).sort_values('gb_importance', ascending=False)
fi_df3.head(10)

## Technique 4 — Permutation Importance (Test Set)
Permutation importance is evaluated on the **test set** — this is correct, as it measures importance to generalisation.

In [ ]:
perm = permutation_importance(rf, X_test_enc, y_test, n_repeats=20, random_state=42, n_jobs=-1)

fi_df4 = pd.DataFrame({
    'feature': feature_names,
    'perm_importance': perm.importances_mean,
    'perm_std': perm.importances_std
}).sort_values('perm_importance', ascending=False)

# FIX: flag negative importance — these features actively hurt generalisation
neg = fi_df4[fi_df4['perm_importance'] < 0]
if len(neg):
    print(f"WARNING: {len(neg)} features with negative permutation importance (may add noise):")
    print(neg[['feature','perm_importance']].to_string())
fi_df4.head(10)

## Technique 5 — LASSO Coefficients

In [ ]:
# FIX: use LassoCV to auto-tune alpha (original used alpha=0.01 which may not be optimal)
lasso = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso.fit(X_train_enc, y_train)
print(f"LassoCV optimal alpha: {lasso.alpha_:.6f}")

fi_df5 = pd.DataFrame({
    'feature': feature_names,
    'lasso_coeff': np.abs(lasso.coef_)
}).sort_values('lasso_coeff', ascending=False)
fi_df5.head(10)

## Technique 6 — Recursive Feature Elimination

In [ ]:
selector = RFE(RandomForestRegressor(n_estimators=50, random_state=42),
               n_features_to_select=len(feature_names), step=1)
selector.fit(X_train_enc, y_train)   # FIX: train set only

fi_df6 = pd.DataFrame({
    'feature': feature_names,
    'rfe_score': selector.estimator_.feature_importances_
}).sort_values('rfe_score', ascending=False)
fi_df6.head(10)

## Technique 7 — Linear Regression Weights

In [ ]:
lin = LinearRegression()
lin.fit(X_train_enc, y_train)   # FIX: train set only

fi_df7 = pd.DataFrame({
    'feature': feature_names,
    'lin_coeff': np.abs(lin.coef_)
}).sort_values('lin_coeff', ascending=False)
fi_df7.head(10)

## Technique 8 — SHAP Values (Test Sample)
Using a sample of the test set for speed.

In [ ]:
# FIX: compute SHAP on test set (not full dataset), with sampling for speed
sample_size = min(300, len(X_test_enc))
X_shap = X_test_enc[:sample_size]

explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)

fi_df8 = pd.DataFrame({
    'feature': feature_names,
    'shap_score': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_score', ascending=False)
fi_df8.head(10)

## Aggregate & Rank

In [ ]:
# Merge on original X feature names (pre-encoding, for model features that map back)
# For simplicity, summarise top-level feature importance using RF + SHAP on original columns
rf_orig = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_orig.fit(X_train, y_train)

# Use original column names
explainer_orig = shap.TreeExplainer(rf_orig)
shap_orig = np.abs(explainer_orig.shap_values(X_test.values[:300])).mean(axis=0)

summary_df = pd.DataFrame({
    'feature': X.columns,
    'rf_importance': rf_orig.feature_importances_,
    'shap_score': shap_orig
}).sort_values('rf_importance', ascending=False)

print(summary_df.to_string(index=False))

## Feature Impact — Drop Lowest and Compare

In [ ]:
from sklearn.model_selection import cross_val_score

rf_eval = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

score_all = cross_val_score(rf_eval, X_train, y_train, cv=5, scoring='r2').mean()
print(f"CV R2 — all features : {score_all:.4f}")

drop_cols = ['pooja room', 'study room', 'others']
X_reduced = X_train.drop(columns=drop_cols, errors='ignore')
score_red = cross_val_score(rf_eval, X_reduced, y_reduced := y_train, cv=5, scoring='r2').mean()
print(f"CV R2 — without {drop_cols}: {score_red:.4f}")
print(f"R2 delta: {score_all - score_red:.4f} (acceptable if < 0.005)")

In [ ]:
# Final export — drop low-importance features
COLS_TO_DROP = ['pooja room', 'study room', 'others']
export_df = X.drop(columns=COLS_TO_DROP, errors='ignore').copy()
export_df['price'] = y.values
export_df.to_csv('gurgaon_properties_post_feature_selection_v2.csv', index=False)
print(f"Saved — shape: {export_df.shape}")
print("Remaining features:", list(export_df.columns))